# Chapter 3 Lab — Morphology and Word Vectors

Part A: stemming vs. lemmatization. Part B: co-occurrence vectors built by hand, then compared
against a small pretrained embedding model. Uses only NLTK's bundled corpora (no external
download of copyrighted text).

In [ ]:
import nltk
for pkg in ["punkt", "wordnet", "omw-1.4", "gutenberg", "averaged_perceptron_tagger"]:
    try:
        nltk.download(pkg, quiet=True)
    except Exception as exc:
        print(f"{pkg} unavailable; continuing with local fallbacks ({type(exc).__name__})")


## Part A — Stemming vs. Lemmatization

In [ ]:
import nltk
for pkg in ["punkt", "wordnet", "omw-1.4", "gutenberg", "averaged_perceptron_tagger"]:
    try:
        nltk.download(pkg, quiet=True)
    except Exception as exc:
        print(f"{pkg} unavailable; continuing with local fallbacks ({type(exc).__name__})")


## Part B — Co-occurrence matrix from scratch

In [ ]:
from nltk.stem import PorterStemmer, SnowballStemmer, LancasterStemmer, WordNetLemmatizer
from collections import Counter
import numpy as np
import re

words = ["studies", "studying", "played", "better", "geese", "ponies"]
porter, snowball, lancaster, lemm = PorterStemmer(), SnowballStemmer("english"), LancasterStemmer(), WordNetLemmatizer()
lemma_fallback = {"studies": "study", "studying": "study", "played": "play", "better": "better", "geese": "geese", "ponies": "pony"}

def safe_lemma(word):
    try:
        return lemm.lemmatize(word, pos="v")
    except LookupError:
        return lemma_fallback.get(word, word)

for w in words:
    print(f"{w:10s} porter={porter.stem(w):10s} snowball={snowball.stem(w):10s} "
          f"lancaster={lancaster.stem(w):10s} lemma={safe_lemma(w)}")

try:
    from nltk.corpus import gutenberg
    corpus_text = " ".join(gutenberg.words("carroll-alice.txt")[:5000]).lower()
    corpus_mode = "NLTK Gutenberg Alice"
except Exception as exc:
    corpus_text = ("alice met the queen and the king in wonderland. "
                   "the queen greeted alice near the garden. "
                   "the king watched the queen and alice play. " * 120).lower()
    corpus_mode = f"synthetic fallback ({type(exc).__name__})"
print("Corpus mode:", corpus_mode)

tokens = re.findall(r"[a-z]+", corpus_text)
vocab = sorted({t for t, c in Counter(tokens).items() if c >= 2})[:250]
idx = {w: i for i, w in enumerate(vocab)}
M = np.zeros((len(vocab), len(vocab)), dtype=float)
window = 2
for i, tok in enumerate(tokens):
    if tok not in idx:
        continue
    for ctx in tokens[max(0, i-window):i] + tokens[i+1:i+window+1]:
        if ctx in idx:
            M[idx[tok], idx[ctx]] += 1
print("Matrix shape:", M.shape)


In [ ]:
def nearest(word, k=5):
    v = M[idx[word]]
    sims = M @ v / (np.linalg.norm(M, axis=1) * np.linalg.norm(v) + 1e-8)
    top = np.argsort(-sims)[1:k+1]
    return [vocab[i] for i in top]

print(nearest("alice"))
print(nearest("queen"))

## Part C — Compare against a pretrained embedding model

In [ ]:
try:
    import gensim.downloader as api
    model = api.load("glove-wiki-gigaword-50")  # small pretrained model
    print(model.most_similar("queen", topn=5))
    print(model.most_similar(positive=["king", "woman"], negative=["man"], topn=3))
except Exception as e:
    print("gensim model download unavailable offline — see README.", e)

## Exercise

Compare `nearest("alice")` from the hand-built co-occurrence matrix against
`model.most_similar("alice")` from the pretrained embedding (if available). Which neighbours
feel more semantically coherent, and why might that be?